# CEC Quantum Computing — Lecture 7 practice

This practice notebook accompanies Lecture 7 on quantum machine learning.

Topics:
- image encoding for quantum circuits
- quantum kernels as overlaps of feature states
- hybrid quantum-classical training with PennyLane and PyTorch
- nonlinear expressivity via data re-uploading

Authors: Dr. Pavel Sulimov, Prof. Dr. R. M. Fuchslin, Claude Lehmann

Target stack:
- PennyLane >= 0.44
- PyTorch >= 2.5
- scikit-learn
- NumPy
- Matplotlib

All qubits start in $|0\rangle$ by default unless we explicitly prepare another state.


In [ ]:
# If needed:
# %pip install -q pennylane torch scikit-learn matplotlib

import math

import matplotlib.pyplot as plt
import numpy as np
import pennylane as qml
import torch
import torch.nn as nn
from sklearn.datasets import load_digits, make_circles, make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.svm import SVC

np.random.seed(7)
torch.manual_seed(7)
torch.set_default_dtype(torch.float64)

print(f"PennyLane: {qml.__version__}")
print(f"Torch: {torch.__version__}")
print(f"NumPy: {np.__version__}")

## Part 1 — Image encoding in PennyLane

We start with a familiar classical object: a handwritten digit image. The goal is to understand what is gained and lost when we map image intensities into a quantum circuit.

We will use a pooled $4\times4$ version of an `sklearn` digits sample so that the encoding logic stays visible.


In [ ]:
digits = load_digits()
mask = np.isin(digits.target, [0, 1])
images = digits.images[mask] / 16.0
labels = digits.target[mask]

sample_image = images[labels == 1][0]


def average_pool_2x2(image: np.ndarray) -> np.ndarray:
    """Average-pool an 8x8 image into a 4x4 image."""
    return image.reshape(4, 2, 4, 2).mean(axis=(1, 3))


def quadrant_means(image_4x4: np.ndarray) -> np.ndarray:
    """Return four coarse features by averaging 2x2 quadrants."""
    return image_4x4.reshape(2, 2, 2, 2).mean(axis=(1, 3)).flatten()


pooled_image = average_pool_2x2(sample_image)
coarse_features = quadrant_means(pooled_image)

fig, axes = plt.subplots(1, 2, figsize=(8, 3.5))
axes[0].imshow(sample_image, cmap="gray_r")
axes[0].set_title("Original 8x8 digit")
axes[0].axis("off")
axes[1].imshow(pooled_image, cmap="gray_r")
axes[1].set_title("Pooled 4x4 digit")
axes[1].axis("off")
plt.tight_layout()

print("coarse angle features:", np.round(coarse_features, 3))

### Task 1.1 — Compare angle and amplitude encoding

Use the pooled image to build two different encodings.

1. Use four quadrant averages as a shallow angle-encoding example on four qubits.
2. Use all sixteen pooled pixels in amplitude encoding on four qubits.
3. Compare the required qubit counts and the information that is visible after encoding.


In [ ]:
def qubits_for_angle(n_features: int) -> int:
    """Angle encoding uses one qubit per feature."""
    return n_features


def qubits_for_amplitude(n_features: int) -> int:
    """Amplitude encoding uses ceil(log2(n_features)) qubits."""
    return math.ceil(math.log2(n_features))


angle_dev = qml.device("default.qubit", wires=4)
amp_dev = qml.device("default.qubit", wires=4)


@qml.qnode(angle_dev)
def angle_encoder(features: np.ndarray):
    qml.AngleEmbedding(features, wires=range(4), rotation="Y")
    return [qml.expval(qml.Z(i)) for i in range(4)]


@qml.qnode(amp_dev)
def amplitude_encoder(features: np.ndarray):
    qml.AmplitudeEmbedding(features, wires=range(4), normalize=True, pad_with=0.0)
    return qml.probs(wires=range(4))


flat_pooled = pooled_image.flatten()
angle_outputs = np.array(angle_encoder(coarse_features * np.pi))
amplitude_probs = np.array(amplitude_encoder(flat_pooled))
normalized_flat = flat_pooled / np.linalg.norm(flat_pooled)

print("qubits for 16 pooled pixels (angle):", qubits_for_angle(len(flat_pooled)))
print("qubits for 16 pooled pixels (amplitude):", qubits_for_amplitude(len(flat_pooled)))
print("angle outputs:", np.round(angle_outputs, 3))

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].bar(np.arange(4), coarse_features)
axes[0].set_title("4 quadrant features for angle encoding")
axes[0].set_xlabel("feature index")
axes[0].set_ylabel("mean intensity")
axes[1].bar(np.arange(16) - 0.2, normalized_flat**2, width=0.4, label="pixel intensity after norm")
axes[1].bar(np.arange(16) + 0.2, amplitude_probs, width=0.4, label="amplitude probabilities")
axes[1].set_title("Amplitude encoding stores all 16 pooled pixels")
axes[1].set_xlabel("basis index")
axes[1].legend()
plt.tight_layout()

### Task 1.2 — Fixed qubit budget

Assume you have only four qubits. Compare how many raw image features can be represented by angle, dense-angle, and amplitude encoding.

This is not a statement about which method is always better. It only highlights the trade-off between qubit count and state-preparation complexity.


In [ ]:
encodings = ["angle", "dense-angle", "amplitude"]
features_per_4_qubits = [4, 8, 16]

plt.figure(figsize=(6, 3.5))
plt.bar(encodings, features_per_4_qubits, color=["#00508C", "#007EA7", "#2E7D32"])
plt.ylabel("raw features represented")
plt.title("Capacity under a 4-qubit budget")
plt.tight_layout()

print("The amplitude bar counts available amplitudes, not implementation ease.")

## Part 2 — Quantum kernel demo

We now make the kernel workflow explicit.

Given two data points $x$ and $y$, we build a feature map $U_\phi(x)$ and define

$$
K(x,y)=\left|\langle 0|^{\otimes n} U_\phi^\dagger(y) U_\phi(x) |0\rangle^{\otimes n}\right|^2.
$$

This is the probability of measuring the all-zero bitstring after:

1. uploading $x$ into the feature map,
2. applying the inverse feature map for $y$,
3. measuring the circuit.

Below we use a small two-qubit ZZ-style feature map so that the whole procedure stays transparent.


In [ ]:
X_circles, y_circles = make_circles(n_samples=90, noise=0.08, factor=0.35, random_state=7)
scaler_circles = MinMaxScaler(feature_range=(0.0, np.pi))
X_circles = scaler_circles.fit_transform(X_circles)
X_train_k, X_test_k, y_train_k, y_test_k = train_test_split(
    X_circles, y_circles, test_size=0.25, random_state=7, stratify=y_circles
)

plt.figure(figsize=(4.2, 4.0))
plt.scatter(X_train_k[:, 0], X_train_k[:, 1], c=y_train_k, cmap="coolwarm", edgecolor="k")
plt.title("Training data for the kernel demo")
plt.xlabel("x1")
plt.ylabel("x2")
plt.tight_layout()

kernel_dev = qml.device("default.qubit", wires=2)


def zz_feature_map(x: np.ndarray):
    """A tiny ZZ-style feature map with single-feature and pairwise terms."""
    qml.Hadamard(wires=0)
    qml.Hadamard(wires=1)
    qml.RZ(2 * x[0], wires=0)
    qml.RZ(2 * x[1], wires=1)
    qml.IsingZZ(2 * (np.pi - x[0]) * (np.pi - x[1]), wires=[0, 1])


@qml.qnode(kernel_dev)
def feature_state(x: np.ndarray):
    zz_feature_map(x)
    return qml.state()


@qml.qnode(kernel_dev)
def overlap_probability(x: np.ndarray, y: np.ndarray):
    zz_feature_map(x)
    qml.adjoint(zz_feature_map)(y)
    return qml.probs(wires=range(2))


def quantum_kernel(x: np.ndarray, y: np.ndarray) -> float:
    sx = feature_state(x)
    sy = feature_state(y)
    overlap_from_states = float(np.abs(np.vdot(sx, sy)) ** 2)
    overlap_from_circuit = float(overlap_probability(x, y)[0])
    assert np.isclose(overlap_from_states, overlap_from_circuit, atol=1e-10)
    return overlap_from_circuit


def kernel_matrix(XA: np.ndarray, XB: np.ndarray) -> np.ndarray:
    return np.array([[quantum_kernel(a, b) for b in XB] for a in XA])


example_overlap = overlap_probability(X_train_k[0], X_train_k[1])
print("Probability distribution for one overlap circuit:", np.round(example_overlap, 4))
print("Kernel entry K(x0, x1) = P(00):", round(float(example_overlap[0]), 4))

K_train = kernel_matrix(X_train_k, X_train_k)
K_test = kernel_matrix(X_test_k, X_train_k)

qsvc = SVC(kernel="precomputed")
qsvc.fit(K_train, y_train_k)
quantum_acc = qsvc.score(K_test, y_test_k)

linear_svc = SVC(kernel="linear")
linear_svc.fit(X_train_k, y_train_k)
linear_acc = linear_svc.score(X_test_k, y_test_k)

print(f"quantum kernel accuracy: {quantum_acc:.3f}")
print(f"linear SVM accuracy: {linear_acc:.3f}")

fig, axes = plt.subplots(1, 2, figsize=(9, 3.8))
im = axes[0].imshow(K_train, cmap="viridis")
axes[0].set_title("Quantum kernel matrix (train)")
axes[0].set_xlabel("j")
axes[0].set_ylabel("i")
plt.colorbar(im, ax=axes[0], label="kernel value")
axes[1].bar(["quantum kernel", "linear SVM"], [quantum_acc, linear_acc], color=["#007EA7", "#C94040"])
axes[1].set_ylim(0.0, 1.05)
axes[1].set_title("Classifier comparison")
axes[1].set_ylabel("test accuracy")
plt.tight_layout()

## Part 3 — Hybrid quantum-classical model with PyTorch

We now place a quantum layer inside a small PyTorch model. The dataset is intentionally tiny so that the whole training loop stays transparent.

The model uses a PennyLane `QNode` as a differentiable feature extractor and a classical dense head for the final prediction.


In [ ]:
X_moons, y_moons = make_moons(n_samples=220, noise=0.15, random_state=7)
moon_scaler = MinMaxScaler(feature_range=(-np.pi / 2, np.pi / 2))
X_moons = moon_scaler.fit_transform(X_moons)
X_train_h, X_test_h, y_train_h, y_test_h = train_test_split(
    X_moons, y_moons, test_size=0.25, random_state=7, stratify=y_moons
)

X_train_t = torch.tensor(X_train_h)
X_test_t = torch.tensor(X_test_h)
y_train_t = torch.tensor(y_train_h, dtype=torch.float64)
y_test_t = torch.tensor(y_test_h, dtype=torch.float64)

hybrid_dev = qml.device("default.qubit", wires=2)


@qml.qnode(hybrid_dev, interface="torch")
def qlayer(inputs, weights):
    qml.AngleEmbedding(inputs, wires=range(2), rotation="Y")
    qml.StronglyEntanglingLayers(weights, wires=range(2))
    return [qml.expval(qml.Z(0)), qml.expval(qml.Z(1))]


weight_shapes = {"weights": (2, 2, 3)}


class HybridClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.quantum = qml.qnn.TorchLayer(qlayer, weight_shapes)
        self.head = nn.Sequential(
            nn.Linear(2, 8),
            nn.ReLU(),
            nn.Linear(8, 1),
        )

    def forward(self, x):
        q_out = self.quantum(x)
        return self.head(q_out).squeeze(-1)


model = HybridClassifier()
optimizer = torch.optim.Adam(model.parameters(), lr=0.05)
loss_fn = nn.BCEWithLogitsLoss()
loss_history = []

for epoch in range(35):
    optimizer.zero_grad()
    logits = model(X_train_t)
    loss = loss_fn(logits, y_train_t)
    loss.backward()
    optimizer.step()
    loss_history.append(float(loss))

with torch.no_grad():
    train_probs = torch.sigmoid(model(X_train_t))
    test_probs = torch.sigmoid(model(X_test_t))
    train_acc = ((train_probs >= 0.5).double() == y_train_t).double().mean().item()
    test_acc = ((test_probs >= 0.5).double() == y_test_t).double().mean().item()

print(f"hybrid train accuracy: {train_acc:.3f}")
print(f"hybrid test accuracy: {test_acc:.3f}")

plt.figure(figsize=(6, 3.5))
plt.plot(loss_history)
plt.xlabel("epoch")
plt.ylabel("binary cross-entropy loss")
plt.title("Hybrid model training curve")
plt.tight_layout()

## Part 4 — Nonlinearity mini-lab via measurement and data re-uploading

We connect this section directly to the lecture theory.

1. First we verify the simplest example of effective nonlinearity:

$$
|\phi(x)\rangle = R_Y(x)|0\rangle, \qquad \langle Z \rangle = \cos(x).
$$

2. Then we compare one re-upload with three re-uploads on a target function that contains higher harmonics.

The goal is to see that repeated data injection increases expressive power without increasing the number of qubits.


In [ ]:
x_reg = torch.linspace(-math.pi, math.pi, 80)
y_reg = torch.sin(x_reg) + 0.35 * torch.sin(3 * x_reg)

one_qubit_dev = qml.device("default.qubit", wires=1)


@qml.qnode(one_qubit_dev)
def one_qubit_expectation(x):
    qml.RY(x, wires=0)
    return qml.expval(qml.Z(0))


analytic_curve = np.cos(x_reg.numpy())
measured_curve = np.array([one_qubit_expectation(float(x)) for x in x_reg])


def make_reupload_model(num_reuploads: int) -> nn.Module:
    dev = qml.device("default.qubit", wires=1)

    @qml.qnode(dev, interface="torch")
    def circuit(x, weights):
        for layer in range(num_reuploads):
            qml.RY(x, wires=0)
            qml.Rot(weights[layer, 0], weights[layer, 1], weights[layer, 2], wires=0)
        return qml.expval(qml.Z(0))

    class ReuploadRegressor(nn.Module):
        def __init__(self):
            super().__init__()
            self.weights = nn.Parameter(0.1 * torch.randn(num_reuploads, 3))

        def forward(self, x_batch):
            return torch.stack([circuit(x, self.weights) for x in x_batch])

    return ReuploadRegressor()


def train_reupload(num_reuploads: int, epochs: int = 120, lr: float = 0.08):
    model = make_reupload_model(num_reuploads)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()
    history = []

    for _ in range(epochs):
        optimizer.zero_grad()
        preds = model(x_reg)
        loss = loss_fn(preds, y_reg)
        loss.backward()
        optimizer.step()
        history.append(float(loss))

    with torch.no_grad():
        final_preds = model(x_reg).detach().numpy()

    return final_preds, history


pred_l1, hist_l1 = train_reupload(1)
pred_l3, hist_l3 = train_reupload(3)

fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
axes[0].plot(x_reg.numpy(), analytic_curve, label="cos(x)", linewidth=2)
axes[0].plot(x_reg.numpy(), measured_curve, "--", label="measured <Z>")
axes[0].set_title("One-qubit nonlinearity")
axes[0].legend()
axes[1].plot(x_reg.numpy(), y_reg.numpy(), label="target", linewidth=2)
axes[1].plot(x_reg.numpy(), pred_l1, label="1 re-upload")
axes[1].plot(x_reg.numpy(), pred_l3, label="3 re-uploads")
axes[1].set_title("Function fit")
axes[1].legend()
axes[2].plot(hist_l1, label="1 re-upload")
axes[2].plot(hist_l3, label="3 re-uploads")
axes[2].set_title("Training loss")
axes[2].set_xlabel("epoch")
axes[2].legend()
plt.tight_layout()

## References

- IBM Quantum Learning, *Introduction to Quantum Machine Learning*.
- IBM Quantum Learning, *Data encoding*.
- IBM Quantum Learning, *Quantum kernel methods*.
- IBM Quantum Learning, *QVCs and QNNs*.
- Maria Schuld and Francesco Petruccione, *Machine Learning with Quantum Computers*.
- Adrián Pérez-Salinas et al., *Data re-uploading for a universal quantum classifier*.
